# Vera — Quickstart Notebook

Boot the system, confirm it's healthy, list capabilities, and call a few.

The orchestrator is expected at **http://localhost:8999**. Start it first with
`make up` (docker) or `make run` (native).

## 1. Configure

In [ ]:
BASE = "http://localhost:8999"
import json, time, urllib.request

def call(method, path, body=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(BASE + path, data=data, method=method,
                                 headers={'content-type': 'application/json'})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read().decode())

def GET(p):        return call('GET', p)
def POST(p, body): return call('POST', p, body)

## 2. Is it up? (optionally start it)

If you have docker and the orchestrator isn't running, uncomment the
`docker compose up -d` line to start the whole stack from the notebook.

In [ ]:
# import subprocess; subprocess.run(['docker','compose','up','-d'])

def wait_until_up(timeout=120):
    start = time.time()
    while time.time() - start < timeout:
        try:
            return GET('/health')
        except Exception as e:
            print('waiting for orchestrator ...', e); time.sleep(3)
    raise RuntimeError('orchestrator did not come up in time')

health = wait_until_up()
health

## 3. Backend health at a glance

In [ ]:
for k in ('redis','postgres','chroma','neo4j'):
    print(f"{'OK ' if health.get(k) else '-- '} {k}")
print('caps:', health.get('caps'), '| workers:', health.get('workers'),
      '| mode:', health.get('mode'))

## 4. List capabilities

Every capability is an MCP tool. `/mcp/tools` returns them all.

In [ ]:
tools = GET('/mcp/tools')
tools = tools['tools'] if isinstance(tools, dict) and 'tools' in tools else tools
names = [t.get('name') for t in tools]
print(len(names), 'capabilities')
names[:40]

## 5. Call capabilities

Invoke any cap via `POST /mcp/call` with `{name, arguments}`.

In [ ]:
def cap(name, **arguments):
    return POST('/mcp/call', {'name': name, 'arguments': arguments})

# Demo calls (safe, no side effects):
cap('health.check')
cap('echo', message='Hello from Vera')
cap('obs.modules')

## 6. Loaded modules

See which capability modules loaded and how many caps each contributed.

In [ ]:
mods = GET('/modules')
for m in (mods if isinstance(mods, list) else mods.get('modules', [])):
    print(f"{m.get('status','?'):>6}  {m.get('name'):<28} +{m.get('caps_added',0)}")

## 7. Watch the event stream

Recent events emitted by capability calls (Redis-backed).

In [ ]:
try:
    print(json.dumps(GET('/events'), indent=2)[:2000])
except Exception as e:
    print('events endpoint unavailable:', e)

---
**Next:** open the harness UI at `http://localhost:8999/`, the API docs at `http://localhost:8999/docs`, or run the guided tour with `python welcome/welcome.py`.